<a href="https://colab.research.google.com/github/khanhtran94/crawTTV/blob/master/chat_gpt_toi_uu_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import copy
import random
import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
TOTAL_NUMBERS = 55
NUM_PICK = 6
SEQ_LEN = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Thiết bị:", device)
print("PyTorch:", torch.__version__)

Thiết bị: cuda
PyTorch: 2.11.0+cu128


In [4]:
URL = (
    "https://raw.githubusercontent.com/khanhtran94/vietlott-data/refs/heads/main/data/power655.jsonl"
)

print("Đang tải dữ liệu...")
df = pd.read_json(URL, lines=True, dtype={"id": str})

required_columns = {"date", "id", "result"}
missing = required_columns - set(df.columns)

if missing:
    raise ValueError(f"Thiếu các cột: {missing}")

if df[["date", "id", "result"]].isna().any().any():
    raise ValueError("Có bản ghi thiếu date, id hoặc result.")

df["date"] = pd.to_datetime(df["date"], errors="raise")

if df["id"].duplicated().any():
    duplicate_ids = df.loc[df["id"].duplicated(), "id"].tolist()
    raise ValueError(f"Trùng mã kỳ quay: {duplicate_ids[:10]}")

df = df.sort_values("date").reset_index(drop=True)

data = []

for _, row in df.iterrows():
    result = row["result"]

    if not isinstance(result, (list, tuple, np.ndarray)) or len(result) != 7:
        raise ValueError(f"Kỳ {row['id']}: result phải có 7 số.")

    values = np.asarray(result, dtype=float)

    if (
        not np.isfinite(values).all()
        or not np.equal(values, np.floor(values)).all()
        or np.any((values < 1) | (values > TOTAL_NUMBERS))
        or len(np.unique(values)) != 7
    ):
        raise ValueError(f"Kỳ {row['id']}: kết quả không hợp lệ: {result}")

    # Lấy 6 số chính trước; bỏ số đặc biệt thứ 7.
    data.append(values[:NUM_PICK].astype(np.int64).tolist())

if len(data) < 100:
    raise ValueError("Cần ít nhất 100 kỳ cho cách chia dữ liệu ở notebook này.")

print(f"Đã tải {len(data)} kỳ quay.")
print("Từ:", df["date"].min().date())
print("Đến:", df["date"].max().date())

print("\n5 kỳ mới nhất:")
print(df[["date", "id", "result"]].tail(5).to_string(index=False))

Đang tải dữ liệu...
Đã tải 1402 kỳ quay.
Từ: 2017-08-01
Đến: 2026-09-24

5 kỳ mới nhất:
      date    id                       result
2026-09-15 01398 [24, 27, 32, 36, 38, 47, 52]
2026-09-17 01399  [6, 11, 25, 27, 37, 45, 15]
2026-09-19 01400   [4, 7, 11, 18, 22, 25, 50]
2026-09-22 01401    [1, 3, 9, 11, 41, 46, 10]
2026-09-24 01402  [1, 13, 23, 25, 26, 28, 27]


In [5]:
draws = np.asarray(data, dtype=np.int64)

encoded = np.zeros(
    (len(draws), TOTAL_NUMBERS),
    dtype=np.float32,
)

encoded[np.arange(len(draws))[:, None], draws - 1] = 1.0

X = np.stack([
    encoded[i - SEQ_LEN:i]
    for i in range(SEQ_LEN, len(encoded))
])

y = encoded[SEQ_LEN:].copy()

# Ngày của kỳ cần dự đoán trong mỗi mẫu.
target_dates = df["date"].iloc[SEQ_LEN:].reset_index(drop=True)

n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Giữ nguyên thứ tự thời gian.
X_train = torch.from_numpy(X[:train_end]).to(device)
y_train = torch.from_numpy(y[:train_end]).to(device)

X_val = torch.from_numpy(X[train_end:val_end]).to(device)
y_val = torch.from_numpy(y[train_end:val_end]).to(device)

X_test = torch.from_numpy(X[val_end:]).to(device)
y_test = torch.from_numpy(y[val_end:]).to(device)

for name, start, end in [
    ("Train", 0, train_end),
    ("Validation", train_end, val_end),
    ("Test", val_end, n),
]:
    print(
        f"{name}: {end - start} mẫu | "
        f"{target_dates.iloc[start].date()} → "
        f"{target_dates.iloc[end - 1].date()}"
    )

print("\nKích thước X:", X.shape)
print("Kích thước y:", y.shape)

Train: 977 mẫu | 2017-08-12 → 2024-01-11
Validation: 210 mẫu | 2024-01-13 → 2025-05-20
Test: 210 mẫu | 2025-05-22 → 2026-09-24

Kích thước X: (1397, 5, 55)
Kích thước y: (1397, 55)


In [6]:
HIDDEN_SIZE = 16
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

class LSTMModel(nn.Module):
    def __init__(self, input_size=55, hidden_size=16):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, input_size)

        # Khởi tạo bias gần xác suất cơ sở 6/55.
        base_probability = NUM_PICK / TOTAL_NUMBERS
        base_logit = math.log(
            base_probability / (1 - base_probability)
        )
        nn.init.constant_(self.fc.bias, base_logit)

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.fc(output[:, -1, :])


def create_model():
    return LSTMModel(
        input_size=TOTAL_NUMBERS,
        hidden_size=HIDDEN_SIZE,
    ).to(device)


model = create_model()
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

parameter_count = sum(p.numel() for p in model.parameters())
print(f"Số tham số: {parameter_count:,}")

Số tham số: 5,607


In [7]:
MAX_EPOCHS = 200
PATIENCE = 15
MIN_DELTA = 1e-5

best_val_loss = float("inf")
best_state = None
best_epoch = 0

# Mốc riêng để kiểm tra cải thiện đủ lớn.
patience_reference = float("inf")
wait = 0

history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    train_logits = model(X_train)
    train_loss = criterion(train_logits, y_train)

    train_loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val), y_val).item()

    if not np.isfinite(train_loss.item()) or not np.isfinite(val_loss):
        raise RuntimeError("Loss không hợp lệ; cần kiểm tra dữ liệu hoặc learning rate.")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss.item(),
        "val_loss": val_loss,
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())

    if val_loss < patience_reference - MIN_DELTA:
        patience_reference = val_loss
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Train: {train_loss.item():.6f} | "
            f"Validation: {val_loss:.6f}"
        )

    if wait >= PATIENCE:
        print(f"\nEarly stopping tại epoch {epoch}.")
        break

model.load_state_dict(best_state)
model.eval()

print(f"\nĐã khôi phục mô hình ở epoch {best_epoch}.")
print(f"Validation loss tốt nhất: {best_val_loss:.6f}")

Epoch   1 | Train: 0.344913 | Validation: 0.344898
Epoch  10 | Train: 0.344474 | Validation: 0.344739
Epoch  20 | Train: 0.344100 | Validation: 0.344680
Epoch  30 | Train: 0.343755 | Validation: 0.344692

Early stopping tại epoch 33.

Đã khôi phục mô hình ở epoch 23.
Validation loss tốt nhất: 0.344677


In [8]:
model.eval()

with torch.no_grad():
    lstm_test_probs = torch.sigmoid(model(X_test)).cpu().numpy()

test_targets = y_test.cpu().numpy()

# Những kỳ quay đã biết đến hết tập train.
# Không lấy dữ liệu validation hoặc test để tính tần suất.
train_draw_history = encoded[:SEQ_LEN + train_end]

# Làm mượt nhẹ bằng prior có trung bình 6/55.
prior_strength = 10.0
base_probability = NUM_PICK / TOTAL_NUMBERS

frequency_probs = (
    train_draw_history.sum(axis=0)
    + prior_strength * base_probability
) / (len(train_draw_history) + prior_strength)

uniform_test_probs = np.full_like(
    test_targets,
    base_probability,
)

frequency_test_probs = np.broadcast_to(
    frequency_probs,
    test_targets.shape,
)


def probability_metrics(probabilities, targets):
    p = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        1e-7,
        1 - 1e-7,
    )

    bce = -np.mean(
        targets * np.log(p)
        + (1 - targets) * np.log1p(-p)
    )

    brier = np.mean((p - targets) ** 2)

    return {
        "BCE": float(bce),
        "Brier": float(brier),
    }


report = pd.DataFrame({
    "Đồng đều": probability_metrics(
        uniform_test_probs, test_targets
    ),
    "Tần suất train": probability_metrics(
        frequency_test_probs, test_targets
    ),
    "LSTM": probability_metrics(
        lstm_test_probs, test_targets
    ),
}).T

print("BCE và Brier: nhỏ hơn là tốt hơn.\n")
print(report.round(6).to_string())

# Top 6 dùng để đánh giá thứ hạng, không phải vé lấy mẫu.
top6_indices = np.argsort(lstm_test_probs, axis=1)[:, -NUM_PICK:]

hits = np.take_along_axis(
    test_targets,
    top6_indices,
    axis=1,
).sum(axis=1).astype(int)

print(f"\nSố kỳ test: {len(hits)}")
print(f"Top 6 LSTM trúng trung bình: {hits.mean():.3f} số/kỳ")
print(f"Kỳ vọng chọn ngẫu nhiên:    {NUM_PICK**2 / TOTAL_NUMBERS:.3f} số/kỳ")

print("\nPhân bố số trúng của top 6 LSTM:")
for hit_count in range(NUM_PICK + 1):
    print(
        f"Trúng {hit_count} số: "
        f"{np.count_nonzero(hits == hit_count)} kỳ"
    )

BCE và Brier: nhỏ hơn là tốt hơn.

                     BCE     Brier
Đồng đều        0.344610  0.097190
Tần suất train  0.344538  0.097175
LSTM            0.344721  0.097210

Số kỳ test: 210
Top 6 LSTM trúng trung bình: 0.690 số/kỳ
Kỳ vọng chọn ngẫu nhiên:    0.655 số/kỳ

Phân bố số trúng của top 6 LSTM:
Trúng 0 số: 100 kỳ
Trúng 1 số: 79 kỳ
Trúng 2 số: 27 kỳ
Trúng 3 số: 4 kỳ
Trúng 4 số: 0 kỳ
Trúng 5 số: 0 kỳ
Trúng 6 số: 0 kỳ


In [9]:
# Khởi tạo lại để không nối tiếp trọng số của mô hình đánh giá.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

final_model = create_model()

final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

X_all = torch.from_numpy(X).to(device)
y_all = torch.from_numpy(y).to(device)

print(f"Huấn luyện toàn bộ lịch sử trong {best_epoch} epoch...")

for epoch in range(1, best_epoch + 1):
    final_model.train()
    final_optimizer.zero_grad(set_to_none=True)

    loss = criterion(final_model(X_all), y_all)

    loss.backward()
    nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
    final_optimizer.step()

final_model.eval()

last_sequence = torch.from_numpy(
    encoded[-SEQ_LEN:].copy()
).unsqueeze(0).to(device)

with torch.no_grad():
    next_probs = torch.sigmoid(
        final_model(last_sequence)
    ).cpu().numpy()[0]

print("Đã tạo trọng số cho kỳ tiếp theo sau:",
      df["date"].iloc[-1].date())

print(f"Tổng 55 xác suất: {next_probs.sum():.3f}")
print("Giá trị tham chiếu là 6; mô hình không ép tổng bằng 6.")

Huấn luyện toàn bộ lịch sử trong 23 epoch...
Đã tạo trọng số cho kỳ tiếp theo sau: 2026-09-24
Tổng 55 xác suất: 6.095
Giá trị tham chiếu là 6; mô hình không ép tổng bằng 6.


In [10]:
def sample_tickets(
    probabilities,
    num_tickets=3,
    temperature=1.0,
    seed=42,
):
    probs = np.asarray(probabilities, dtype=np.float64)

    if probs.shape != (TOTAL_NUMBERS,):
        raise ValueError("Cần đúng 55 xác suất.")

    if (
        not np.isfinite(probs).all()
        or np.any(probs < 0)
        or np.any(probs > 1)
        or probs.sum() <= 0
    ):
        raise ValueError("Xác suất không hợp lệ.")

    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature phải hữu hạn và > 0.")

    if not isinstance(num_tickets, int) or num_tickets <= 0:
        raise ValueError("num_tickets phải là số nguyên dương.")

    log_weights = np.log(np.clip(probs, 1e-12, 1.0))
    log_weights -= log_weights.max()

    # Chặn mức quá nhỏ để tránh underflow.
    scaled_log_weights = np.maximum(
        log_weights / temperature, -700
    )
    weights = np.exp(scaled_log_weights)
    weights /= weights.sum()

    rng = np.random.default_rng(seed)

    tickets = []
    seen = set()

    for _ in range(10_000):
        chosen = rng.choice(
            np.arange(1, TOTAL_NUMBERS + 1),
            size=NUM_PICK,
            replace=False,
            p=weights,
        )

        ticket = tuple(sorted(int(number) for number in chosen))

        if ticket not in seen:
            seen.add(ticket)
            tickets.append(list(ticket))

        if len(tickets) == num_tickets:
            return tickets

    raise RuntimeError(
        "Không tạo đủ vé khác nhau. "
        "Hãy tăng temperature hoặc giảm số vé."
    )


TEMPERATURE = 1.0
TICKET_SEED = 42

tickets = sample_tickets(
    probabilities=next_probs,
    num_tickets=3,
    temperature=TEMPERATURE,
    seed=TICKET_SEED,
)

print(f"\n3 vé lấy mẫu — Temperature {TEMPERATURE}")

for index, ticket in enumerate(tickets, start=1):
    formatted = " - ".join(f"{number:02d}" for number in ticket)
    print(f"Vé {index}: {formatted}")


3 vé lấy mẫu — Temperature 1.0
Vé 1: 06 - 25 - 39 - 43 - 48 - 54
Vé 2: 08 - 21 - 25 - 42 - 44 - 51
Vé 3: 04 - 13 - 25 - 31 - 36 - 46


In [11]:
def check_winning_ticket(winning_numbers, generated_tickets):
    winning_set = set(winning_numbers)

    if (
        len(winning_numbers) != 6
        or len(winning_set) != 6
        or not all(isinstance(n, (int, np.integer)) and 1 <= n <= 55
                   for n in winning_numbers)
    ):
        raise ValueError("Kết quả phải gồm 6 số nguyên khác nhau từ 1 đến 55.")

    print("\n" + "=" * 45)
    print("KIỂM TRA SỐ TRÙNG CỦA CÁC VÉ")
    print(f"6 số chính thực tế: {sorted(winning_set)}")
    print("=" * 45)

    results = []

    for index, ticket in enumerate(generated_tickets, start=1):
        common_numbers = sorted(winning_set.intersection(ticket))

        print(f"\nVé {index}: {ticket}")
        print(f"  → Trùng {len(common_numbers)}/6 số")
        print(f"  → Các số trùng: {common_numbers or 'Không có'}")

        results.append({
            "id": index,
            "ticket": list(ticket),
            "matched_numbers": common_numbers,
            "num_matches": len(common_numbers),
        })

    return results


# Thay bằng 6 số chính thực tế của kỳ bạn muốn kiểm tra.
# Đây chỉ là dữ liệu ví dụ, không xác nhận là kết quả kỳ 1402.
winning_numbers_example = [14, 18, 21, 38, 48, 52]

# tickets là 3 vé đã sinh tại ô 8.
check_results = check_winning_ticket(winning_numbers_example, tickets)


KIỂM TRA SỐ TRÙNG CỦA CÁC VÉ
6 số chính thực tế: [14, 18, 21, 38, 48, 52]

Vé 1: [6, 25, 39, 43, 48, 54]
  → Trùng 1/6 số
  → Các số trùng: [48]

Vé 2: [8, 21, 25, 42, 44, 51]
  → Trùng 1/6 số
  → Các số trùng: [21]

Vé 3: [4, 13, 25, 31, 36, 46]
  → Trùng 0/6 số
  → Các số trùng: Không có
